# Toxicity and Sentiment Scoring

Scores the cleaned reviews with Detoxify for toxicity and VADER for sentiment, and writes one parquet per game.

# Setup and the GPU

In [1]:
import json
import time
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from detoxify import Detoxify
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
clean_reviews_dir=project_root/'data'/'processed'/'interim'/'reviews'
toxicity_dir=project_root/'data'/'processed'/'interim'/'toxicity'
tables_dir=project_root/'data'/'processed'/'tables'
toxicity_dir.mkdir(parents=True,exist_ok=True)
tables_dir.mkdir(parents=True,exist_ok=True)

# fixed seed so the sampled reviews are the same on every re-run
random_seed=42
np.random.seed(random_seed)

print('clean reviews in :',clean_reviews_dir)
print('scores go to     :',toxicity_dir)
print()
print('cuda available :',torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu            :',torch.cuda.get_device_name(0))
    print('vram total     :',round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'gb')
scoring_device='cuda' if torch.cuda.is_available() else 'cpu'

# vader is a lexicon lookup, so loading it costs nothing
sentiment_analyser=SentimentIntensityAnalyzer()

clean reviews in : d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis\data\processed\interim\reviews
scores go to     : d:\Msc_Data_Analytics\Research_Practicum_final\final_thesis\data\processed\interim\toxicity

cuda available : True
gpu            : NVIDIA GeForce RTX 4050 Laptop GPU
vram total     : 6.4 gb


# Deciding which games i am allowed to score

A game is scorable if the collector resolved a launch date, or if it is still in early access.

In [2]:
checkpoint_dir=project_root/'data'/'raw'/'_checkpoints'


def read_checkpoint(appid):
    checkpoint_file=checkpoint_dir/f'{appid}.json'
    if not checkpoint_file.exists():
        return {}
    try:
        return json.loads(checkpoint_file.read_text())
    except json.JSONDecodeError:
        return {}


game_status=[]
for clean_file_path in sorted(clean_reviews_dir.glob('*.parquet')):
    appid=int(clean_file_path.stem)
    checkpoint=read_checkpoint(appid)
    launch_timestamp=checkpoint.get('derived_launch_timestamp')
    cohort=checkpoint.get('cohort','unknown')

    game_status.append({'appid':appid,
                        'cohort':cohort,
                        'launch_timestamp':launch_timestamp,
                        # scorable with a launch date, or still in early access
                        'will_score':pd.notna(launch_timestamp) or cohort=='still_in_ea'})

df_status=pd.DataFrame(game_status)

# join stage 2 english share, recomputing would read 100%
df_english=pd.read_csv(project_root/'data'/'processed'/'interim'/'clean_rejects.csv')
df_status=df_status.merge(df_english[['appid','n_reviews_all_languages','n_english','english_share','kept']],
                          on='appid',how='left')

print(' GAMES SCORING')
print(df_status['cohort'].value_counts().to_string())
print(f'games cleaned          : {len(df_status)}')
print(f'scoring now            : {df_status["will_score"].sum()}')
print(f'held back              : {(~df_status["will_score"]).sum()} (no launch date and not tagged still_in_ea)')

appids_to_score=df_status.loc[df_status['will_score'],'appid'].tolist()
launch_by_appid=dict(zip(df_status['appid'],df_status['launch_timestamp']))

# a fixed sample for the audit cells, so results regenerate
audit_sample_games=200
audit_sample=sorted(np.random.default_rng(random_seed).choice(
    appids_to_score,size=min(audit_sample_games,len(appids_to_score)),replace=False).tolist())
print(f'audit sample for the end-of-notebook checks: {len(audit_sample)} games (seed {random_seed})')

 GAMES SCORING
cohort
launched       1230
still_in_ea     563
games cleaned          : 1793
scoring now            : 1793
held back              : 0 (no launch date and not tagged still_in_ea)
audit sample for the end-of-notebook checks: 200 games (seed 42)


# English only, and what that costs me

The English share measured in the cleaning stage is carried through as a covariate.

In [3]:
print('LANGUAGE COVERAGE, WHICH IS A CONFOUND AND NOT JUST A FILTER')
df_language=df_status[df_status['will_score']][['appid','n_reviews_all_languages','n_english','english_share']]
print(df_language.to_string(index=False))
print(f'english share across the games i am scoring : '
      f'{100*df_language["n_english"].sum()/df_language["n_reviews_all_languages"].sum():.1f}%')
print(f'lowest english share of any game           : {df_language["english_share"].min()}%')
print(f'highest english share of any game          : {df_language["english_share"].max()}%')
print('that spread is why english_share is carried forward as a per game covariate')
print('rather than treated as a filter i can forget about')

LANGUAGE COVERAGE, WHICH IS A CONFOUND AND NOT JUST A FILTER
  appid  n_reviews_all_languages  n_english  english_share
1000030                      NaN        NaN            NaN
1000080                   1571.0      158.0          10.06
1000360                     77.0       44.0          57.14
1004270                   1731.0      577.0          33.33
1005400                    258.0      173.0          67.05
1005930                   1627.0      213.0          13.09
1010860                    748.0      115.0          15.37
1011290                    212.0      185.0          87.26
1012110                   2306.0     1290.0          55.94
1013320                   6912.0     2488.0          36.00
1015140                   1001.0      569.0          56.84
1016920                   2262.0      928.0          41.03
1018830                   3221.0     1910.0          59.30
1020800                    614.0      181.0          29.48
1020910                    201.0       50.0          2

# The sampling cap and how i spend it

A fixed budget per game, spread evenly across the early-access period so the trajectory features stay usable.

In [4]:
reviews_scored_per_game=2000
cohort_time_bins=8
minimum_per_bin=25


def allocate_across_time(df_cohort,budget,rng):
    """pick `budget` reviews spread across the cohort's lifetime rather than bunched at one end"""
    if len(df_cohort)<=budget:
        return df_cohort

    # equal width bins, so a quiet stretch keeps its own bin
    bins=pd.cut(df_cohort['review_date'],bins=cohort_time_bins,labels=False)
    picked=[]
    bin_sizes=bins.value_counts().sort_index()

    # proportional share, then lift any bin up to the floor
    shares=(budget*bin_sizes/bin_sizes.sum()).round().astype(int)
    shares=shares.clip(upper=bin_sizes)
    shares=np.maximum(shares,np.minimum(minimum_per_bin,bin_sizes))

    # the floor can overshoot the budget, so trim the biggest bins
    while shares.sum()>budget:
        shares[shares.idxmax()]-=1

    for bin_index,take in shares.items():
        if take>0:
            group=df_cohort[bins==bin_index]
            picked.append(group.sample(n=int(take),random_state=rng))
    return pd.concat(picked)


def build_scoring_sample(df_game,appid):
    """choose which reviews of one game get scored, spending the cap across both cohorts"""
    # stage 2 already filtered to english, so nothing to exclude here
    eligible=df_game[df_game['review_text_clean'].str.strip()!=''].copy()

    # cohort is the timestamp against the bisected launch date, not the steam flag
    # pd.notna, not a bare truth test - nan is truthy
    launch_timestamp=launch_by_appid.get(appid)
    if pd.notna(launch_timestamp):
        df_ea=eligible[eligible['timestamp_created']<int(launch_timestamp)]
        df_post=eligible[eligible['timestamp_created']>=int(launch_timestamp)]
    else:
        df_ea,df_post=eligible,eligible.iloc[0:0]

    # half the budget each, with the unused remainder passed to the other cohort
    half=reviews_scored_per_game//2
    ea_budget=min(len(df_ea),half)
    post_budget=min(len(df_post),reviews_scored_per_game-ea_budget)
    ea_budget=min(len(df_ea),reviews_scored_per_game-post_budget)

    rng=random_seed+appid
    parts=[]
    if ea_budget>0:
        parts.append(allocate_across_time(df_ea,ea_budget,rng))
    if post_budget>0:
        parts.append(allocate_across_time(df_post,post_budget,rng))
    return pd.concat(parts) if parts else eligible.head(0)


print('sampling settings')
print(f'cap per game        : {reviews_scored_per_game}')
print(f'time bins per cohort: {cohort_time_bins}')
print(f'floor per bin       : {minimum_per_bin}')
print(f'seed                : {random_seed} (plus the appid, so games are independent)')

sampling settings
cap per game        : 2000
time bins per cohort: 8
floor per bin       : 25
seed                : 42 (plus the appid, so games are independent)


# Scoring

Half precision, with reviews length-sorted before batching and the order restored afterwards.

# Steam censors the reviews before it gives them to me

Steam replaces profanity with hearts, which VADER reads as positive. The cleaning stage tokenises them, so the censored rate is recoverable as its own feature.

In [5]:
import re

# counting the token recovers the profanity signal steam censored
censored_token_pattern=re.compile(r'\[censored\]')


def count_censored_tokens(text):
    """how many times steam redacted profanity in this review"""
    return len(censored_token_pattern.findall(text))


print('why stage 2 tokenises the mask instead of deleting it')
demonstration=[('♥♥♥♥♥♥♥ suck and u should go kil yourself if you plat',
                '[censored] suck and u should go kil yourself if you plat'),
               ('suck me ♥♥♥♥','suck me [censored]'),
               ('this game is genuinely lovely and i adore it','this game is genuinely lovely and i adore it')]
for as_steam_sent_it,as_stage_2_stores_it in demonstration:
    before=sentiment_analyser.polarity_scores(as_steam_sent_it)['compound']
    after=sentiment_analyser.polarity_scores(as_stage_2_stores_it)['compound']
    print(f'  raw {before:+.3f}  ->  cleaned {after:+.3f}   '
          f'({count_censored_tokens(as_stage_2_stores_it)} redaction(s))  {as_stage_2_stores_it[:48]}')
print('the third line is the control - a genuinely positive review has no mask in it and')
print('does not move. only the censored ones move, and they all move downward')
print()
print('one honest caveat: [censored] is not neutral to vader either, it scores -0.153')
print('because "censored" is in the lexicon. that is a small negative nudge on reviews')
print('that did contain profanity, which is defensible, but it is a choice not a fix')

why stage 2 tokenises the mask instead of deleting it
  raw +0.983  ->  cleaned -0.542   (1 redaction(s))  [censored] suck and u should go kil yourself if 
  raw +0.942  ->  cleaned -0.542   (1 redaction(s))  suck me [censored]
  raw +0.813  ->  cleaned +0.813   (0 redaction(s))  this game is genuinely lovely and i adore it
the third line is the control - a genuinely positive review has no mask in it and
does not move. only the censored ones move, and they all move downward

one honest caveat: [censored] is not neutral to vader either, it scores -0.153
because "censored" is in the lexicon. that is a small negative nudge on reviews
that did contain profanity, which is defensible, but it is a choice not a fix


In [6]:
max_characters_scored=2000
scoring_batch_size=64

toxicity_model=Detoxify('original',device=scoring_device)
# half precision, about 3x faster with scores agreeing to three decimals
if scoring_device=='cuda':
    toxicity_model.model=toxicity_model.model.half()
print('detoxify is loaded on',scoring_device)


def score_texts(texts):
    """run both scorers over a list of review strings, returned in the order they came in"""
    # text arrives cleaned, so only count the redaction markers
    mask_counts=[count_censored_tokens(t) for t in texts]
    trimmed=[t[:max_characters_scored] for t in texts]

    # group similar lengths so batches are not padded to one long outlier
    length_order=np.argsort([len(t) for t in trimmed])

    scored_batches=[]
    for start in range(0,len(length_order),scoring_batch_size):
        batch_positions=length_order[start:start+scoring_batch_size]
        with torch.no_grad():
            scored_batches.append(pd.DataFrame(toxicity_model.predict([trimmed[i] for i in batch_positions])))
    df_scores=pd.concat(scored_batches,ignore_index=True)

    # undo the length sort so the scores land back on the right rows
    df_scores.index=length_order
    df_scores=df_scores.sort_index().reset_index(drop=True)

    # vader costs nothing next to bert, so run it inline
    df_scores['sentiment_compound']=[sentiment_analyser.polarity_scores(t)['compound'] for t in trimmed]

    # how much profanity steam censored out of this review
    df_scores['masked_profanity_runs']=mask_counts
    return df_scores


print('checking the length sort actually gets undone before i trust it on real data')

# worst case: the toxic line is shortest, so the sort must move it
order_check_texts=['you are a worthless idiot',
                   'this game is wonderful and the community has been kind to me '*40,
                   'solid crafting loop, worth the money',
                   'the servers are down again which is frustrating but the devs are communicating']
order_check=score_texts(order_check_texts)

for text,toxicity in zip(order_check_texts,order_check['toxicity']):
    print(f'  tox {toxicity:.3f}  <- {text[:60]}')

# row zero going in must be row zero coming out
assert order_check['toxicity'].idxmax()==0,'the length sort was not undone, scores are misaligned'
assert order_check['toxicity'].iloc[0]>0.9,'the known toxic line did not score as toxic'
assert order_check['toxicity'].iloc[1:].max()<0.1,'a clean line scored as toxic'
print('reorder is correct, the toxic line came back on the row it went in on')


def score_one_game(appid):
    """sample, score and cache one game. returns a short record for the run log"""
    output_file_path=toxicity_dir/f'{appid}.parquet'
    if output_file_path.exists():
        return None

    df_game=pd.read_parquet(clean_reviews_dir/f'{appid}.parquet')
    df_sample=build_scoring_sample(df_game,appid)
    if df_sample.empty:
        return {'appid':appid,'n_scored':0,'seconds':0.0}

    started_at=time.time()
    df_scores=score_texts(df_sample['review_text_clean'].tolist())

    # keep only the join and cohort keys, never the review text
    df_scores.insert(0,'review_id',df_sample['review_id'].values)
    df_scores.insert(1,'appid',appid)
    df_scores.insert(2,'written_during_early_access',df_sample['written_during_early_access'].values)
    df_scores.insert(3,'timestamp_created',df_sample['timestamp_created'].values)
    df_scores.insert(3,'review_date',df_sample['review_date'].values)
    df_scores.to_parquet(output_file_path,index=False)

    elapsed=time.time()-started_at
    return {'appid':appid,'n_scored':len(df_scores),'seconds':round(elapsed,1)}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

detoxify is loaded on cuda
checking the length sort actually gets undone before i trust it on real data
  tox 0.986  <- you are a worthless idiot
  tox 0.001  <- this game is wonderful and the community has been kind to me
  tox 0.001  <- solid crafting loop, worth the money
  tox 0.002  <- the servers are down again which is frustrating but the devs
reorder is correct, the toxic line came back on the row it went in on


In [7]:
print('SCORING RUN')

run_log=[]
for position,appid in enumerate(appids_to_score,start=1):
    record=score_one_game(appid)
    if record is None:
        print(f'[{position}/{len(appids_to_score)}] app {appid}: already scored, skipping')
        continue
    run_log.append(record)
    rate=record['n_scored']/record['seconds'] if record['seconds'] else 0
    print(f'[{position}/{len(appids_to_score)}] app {appid}: scored {record["n_scored"]:,} '
          f'in {record["seconds"]}s ({rate:.0f} reviews/sec)')

if run_log:
    total_scored=sum(r['n_scored'] for r in run_log)
    total_seconds=sum(r['seconds'] for r in run_log)
    print(f'scored {total_scored:,} reviews in {total_seconds:.0f}s '
          f'({total_scored/max(total_seconds,1):.0f} reviews/sec)')
    # how long the full catalogue will take
    print(f'at that rate 2,000 games at the {reviews_scored_per_game} cap is about '
          f'{2000*reviews_scored_per_game/max(total_scored/max(total_seconds,1),1)/3600:.1f} hours')

SCORING RUN
[1/1793] app 1000030: already scored, skipping
[2/1793] app 1000080: already scored, skipping
[3/1793] app 1000360: already scored, skipping
[4/1793] app 1004270: already scored, skipping
[5/1793] app 1005400: already scored, skipping
[6/1793] app 1005930: already scored, skipping
[7/1793] app 1010860: already scored, skipping
[8/1793] app 1011290: already scored, skipping
[9/1793] app 1012110: already scored, skipping
[10/1793] app 1013320: already scored, skipping
[11/1793] app 1015140: already scored, skipping
[12/1793] app 1016920: already scored, skipping
[13/1793] app 1018830: already scored, skipping
[14/1793] app 1020800: already scored, skipping
[15/1793] app 1020910: already scored, skipping
[16/1793] app 1022310: already scored, skipping
[17/1793] app 1022450: already scored, skipping
[18/1793] app 1025600: already scored, skipping
[19/1793] app 1026840: already scored, skipping
[20/1793] app 1029210: already scored, skipping
[21/1793] app 1029550: already scored

# Looking at what the model actually did

A read of the most toxic reviews and the score distribution.

In [8]:
# pull the most toxic reviews back with their text so i can read them
eyeball_rows=[]
# the fixed audit sample is enough to surface the failure modes
eyeball_files=[toxicity_dir/f'{appid}.parquet' for appid in audit_sample]
for scored_file_path in eyeball_files:
    df_scored=pd.read_parquet(scored_file_path)
    if df_scored.empty:
        continue
    df_text=pd.read_parquet(clean_reviews_dir/scored_file_path.name,columns=['review_id','review_text_clean'])
    df_joined=df_scored.merge(df_text,on='review_id')
    eyeball_rows.append(df_joined.nlargest(4,'toxicity'))

df_eyeball=pd.concat(eyeball_rows).nlargest(40,'toxicity')
df_eyeball['review_text_clean']=df_eyeball['review_text_clean'].str.replace(r'\s+',' ',regex=True).str.slice(0,150)
df_eyeball[['appid','toxicity','insult','threat','sentiment_compound','review_text_clean']].to_csv(
    tables_dir/'toxicity_eyeball.csv',index=False)

print('THE 25 REVIEWS DETOXIFY IS MOST SURE ARE TOXIC')
for _,row in df_eyeball.head(25).iterrows():
    print(f'tox {row["toxicity"]:.3f} | insult {row["insult"]:.3f} | vader {row["sentiment_compound"]:+.2f}')
    print(f'  {row["review_text_clean"]}')
print('full list written to',tables_dir/'toxicity_eyeball.csv')

THE 25 REVIEWS DETOXIFY IS MOST SURE ARE TOXIC
tox 0.999 | insult 0.961 | vader +0.01
  you can suck you enemies and let ur friends suck you. very cool, very nice
tox 0.998 | insult 0.939 | vader -0.80
  You bastards! Gimme it back! I wanna play more! MORE! I DEMAND IT!
tox 0.998 | insult 0.824 | vader -0.44
  Slap that ass
tox 0.998 | insult 0.857 | vader +0.36
  You can slap my ass like that ball UWU <3
tox 0.998 | insult 0.951 | vader -0.80
  because VAMPIRES! you get to build your castles, you have pvp and/or pve setup for your servers, which is really fun with friends! AND you suck suck s
tox 0.998 | insult 0.746 | vader -0.13
  Nun wont do stuff I will suck her wee wee play pacify
tox 0.998 | insult 0.782 | vader -0.36
  brawlhalla this d i c k because it sucks so much it f uc kin fuc ks
tox 0.998 | insult 0.693 | vader +0.91
  What the f***? What in the honest to F*CKING F*CK did I just F*CKING PLAY! DEAR GOD, WHY IN THE HOLY HELL OF SATAN'S LEFT JUICY A**-PIPE DID I ACCEPT 
tox

In [9]:
# a healthy distribution is skewed to zero
df_all_scores=pd.concat([pd.read_parquet(f) for f in sorted(toxicity_dir.glob('*.parquet'))])

print('SCORE DISTRIBUTION ACROSS EVERYTHING SCORED SO FAR')
print(f'reviews scored : {len(df_all_scores):,}')
print()
for column in ['toxicity','severe_toxicity','insult','threat','identity_attack','sentiment_compound']:
    values=df_all_scores[column]
    print(f'{column:20s} median {values.median():+.4f}  mean {values.mean():+.4f}  '
          f'p95 {values.quantile(0.95):+.4f}  share>0.5 {100*(values>0.5).mean():5.2f}%')

masked=df_all_scores['masked_profanity_runs']>0
print()
print(f'reviews steam censored : {masked.sum():,} ({100*masked.mean():.1f}%)')
print(f'  mean toxicity where steam censored something : {df_all_scores.loc[masked,"toxicity"].mean():.4f}')
print(f'  mean toxicity where it did not               : {df_all_scores.loc[~masked,"toxicity"].mean():.4f}')
print('  the gap is why the mask count is kept as a feature rather than discarded')

print()
print('by cohort, which is the comparison the whole project rests on:')
print(df_all_scores.groupby('written_during_early_access')[['toxicity','insult','sentiment_compound']]
      .mean().round(4).to_string())

SCORE DISTRIBUTION ACROSS EVERYTHING SCORED SO FAR
reviews scored : 1,512,547

toxicity             median +0.0011  mean +0.0442  p95 +0.2891  share>0.5  3.29%
severe_toxicity      median +0.0001  mean +0.0007  p95 +0.0008  share>0.5  0.00%
insult               median +0.0002  mean +0.0068  p95 +0.0131  share>0.5  0.26%
threat               median +0.0001  mean +0.0023  p95 +0.0012  share>0.5  0.12%
identity_attack      median +0.0001  mean +0.0012  p95 +0.0017  share>0.5  0.04%
sentiment_compound   median +0.6369  mean +0.4314  p95 +0.9850  share>0.5 58.01%

reviews steam censored : 53,464 (3.5%)
  mean toxicity where steam censored something : 0.1240
  mean toxicity where it did not               : 0.0412
  the gap is why the mask count is kept as a feature rather than discarded

by cohort, which is the comparison the whole project rests on:
                             toxicity  insult  sentiment_compound
written_during_early_access                                      
False       

# Does the cleaning actually change the scores

In [10]:
# a preprocessing check, not a per game result: does cleaning move the scores
print(f'comparing raw against cleaned text on {len(audit_sample)} randomly chosen games')

comparison_rows=[]

for appid in audit_sample:
    df_game=pd.read_parquet(clean_reviews_dir/f'{appid}.parquet')
    df_sample=build_scoring_sample(df_game,appid)
    if df_sample.empty:
        continue

    scores_on_raw=score_texts(df_sample['review_text_raw'].tolist())
    scores_on_clean=score_texts(df_sample['review_text_clean'].tolist())

    comparison_rows.append({
        'appid':appid,
        'n':len(df_sample),
        'toxicity_raw':scores_on_raw['toxicity'].mean(),
        'toxicity_clean':scores_on_clean['toxicity'].mean(),
        'sentiment_raw':scores_on_raw['sentiment_compound'].mean(),
        'sentiment_clean':scores_on_clean['sentiment_compound'].mean(),
        # correlation matters more than the mean here
        'toxicity_correlation':scores_on_raw['toxicity'].corr(scores_on_clean['toxicity']),
        'sentiment_correlation':scores_on_raw['sentiment_compound'].corr(scores_on_clean['sentiment_compound'])})

df_comparison=pd.DataFrame(comparison_rows)
df_comparison['toxicity_delta']=df_comparison['toxicity_clean']-df_comparison['toxicity_raw']
df_comparison['sentiment_delta']=df_comparison['sentiment_clean']-df_comparison['sentiment_raw']
df_comparison.round(4).to_csv(tables_dir/'raw_vs_clean_delta.csv',index=False)

print('SCORING THE SAME REVIEWS ON RAW TEXT AND ON CLEANED TEXT')
print(f'(random sample of {len(audit_sample)} games, seed {random_seed})')
print(df_comparison[['appid','n','toxicity_raw','toxicity_clean','toxicity_delta',
                     'sentiment_raw','sentiment_clean','sentiment_delta']].head(12).round(4).to_string(index=False))
print(f'... showing 12 of {len(df_comparison)} games')
print(f'mean toxicity  : raw {df_comparison["toxicity_raw"].mean():.4f} -> '
      f'clean {df_comparison["toxicity_clean"].mean():.4f} '
      f'(delta {df_comparison["toxicity_delta"].mean():+.4f})')
print(f'mean sentiment : raw {df_comparison["sentiment_raw"].mean():.4f} -> '
      f'clean {df_comparison["sentiment_clean"].mean():.4f} '
      f'(delta {df_comparison["sentiment_delta"].mean():+.4f})')
print()
print(f'review level correlation between the two, toxicity  : '
      f'{df_comparison["toxicity_correlation"].mean():.4f}')
print(f'review level correlation between the two, sentiment : '
      f'{df_comparison["sentiment_correlation"].mean():.4f}')
print('written to',tables_dir/'raw_vs_clean_delta.csv')

comparing raw against cleaned text on 200 randomly chosen games
SCORING THE SAME REVIEWS ON RAW TEXT AND ON CLEANED TEXT
(random sample of 200 games, seed 42)
 appid    n  toxicity_raw  toxicity_clean  toxicity_delta  sentiment_raw  sentiment_clean  sentiment_delta
215550  202        0.0759          0.0754         -0.0005         0.2205           0.1673          -0.0532
221100 1999        0.1031          0.1020         -0.0011         0.2823           0.1506          -0.1317
223410  552        0.0340          0.0337         -0.0003         0.4782           0.4642          -0.0140
225600 2000        0.0684          0.0675         -0.0009         0.3364           0.2839          -0.0525
228380 1999        0.0389          0.0390          0.0001         0.5620           0.5329          -0.0290
230190 1155        0.0177          0.0178          0.0001         0.5651           0.5457          -0.0194
230860  576        0.0409          0.0408         -0.0001         0.5573           0.5422   

# The assertion cell

In [11]:
scored_files=sorted(toxicity_dir.glob('*.parquet'))
assert len(scored_files)>0,'nothing was scored'

score_columns=['toxicity','severe_toxicity','obscene','threat','insult','identity_attack']

for scored_file_path in scored_files:
    df_check=pd.read_parquet(scored_file_path)
    appid=int(scored_file_path.stem)

    assert len(df_check)<=reviews_scored_per_game,f'{appid} exceeded the sampling cap'
    assert df_check['review_id'].is_unique,f'{appid} scored a review twice'
    assert df_check[score_columns].notna().all().all(),f'{appid} has null scores'
    assert df_check[score_columns].ge(0).all().all() and df_check[score_columns].le(1).all().all(),\
        f'{appid} has scores outside 0 to 1'
    assert df_check['sentiment_compound'].between(-1,1).all(),f'{appid} has a bad vader score'
    assert df_check['masked_profanity_runs'].ge(0).all(),f'{appid} has a negative mask count'

    # every scored review must exist in the clean layer or the stage 4 join breaks
    clean_ids=set(pd.read_parquet(clean_reviews_dir/scored_file_path.name,columns=['review_id'])['review_id'])
    assert set(df_check['review_id']).issubset(clean_ids),f'{appid} scored a review that is not in the clean layer'

# the games held back must not have been scored
held_back=set(df_status.loc[~df_status['will_score'],'appid'])
assert held_back.isdisjoint({int(f.stem) for f in scored_files}),'a held back game was scored'
# every scored game must carry the cohort column
for scored_file_path in scored_files[:50]:
    assert 'timestamp_created' in pd.read_parquet(scored_file_path).columns,        f'{scored_file_path.name} has no timestamp_created - stage 4 cannot split its cohorts'

print(f'all {len(scored_files)} scored files passed')
print(f'{len(df_all_scores):,} reviews carry toxicity and sentiment scores')
print(f'{len(held_back)} games held back - no launch date and not tagged still_in_ea')

all 1793 scored files passed
1,512,547 reviews carry toxicity and sentiment scores
0 games held back - no launch date and not tagged still_in_ea
